# Reflection Pattern

The first pattern we are going to implement is the **reflection pattern**.

---

<img src="../img/reflection_pattern.png" alt="Alt text" width="600"/>

---

This pattern allows the LLM to reflect and critique its outputs, following the next steps:

1. The LLM **generates** a candidate output. If you look at the diagram above, it happens inside the **"Generate"** box.
2. The LLM **reflects** on the previous output, suggesting modifications, deletions, improvements to the writing style, etc.
3. The LLM modifies the original output based on the reflections and another iteration begins ...

**Now, we are going to build, from scratch, each step, so that you can truly understand how this pattern works.**

## Generation Step

The first thing we need to consider is:

> What do we want to generate? A poem? An essay? Python code?

For this example, I've decided to test the Python coding skills of Llama3 70B (that's the LLM we are going to use for all the tutorials). In particular, we are going to ask our LLM to code a famous sorting algorithm: **Merge Sort**.

---

<img src="../img/mergesort.png" alt="Alt text" width="500"/>

### Groq Client and relevant imports

In [1]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.4 MB/s eta 0:00:00


In [2]:
import os
from pprint import pprint
from groq import Groq
from IPython.display import display_markdown

# Set the environment variable directly
os.environ["GROQ_API_KEY"] = ""

# The client will now automatically pick up the key from os.environ
client = Groq()

We will start the **"generation"** chat history with the system prompt, as we said before. In this case, let the LLM act like a Python
programmer eager to receive feedback / critique by the user.

In [3]:
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique,"
        "respond with a revised version of your previous attempt."
    }
]

Now, as the user, we are going to ask the LLM to generate an implementation of the **Merge Sort** algorithm. Just add a new message with the **user** role to the chat history.

In [4]:
generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Merge Sort algorithm"
    }
)

Let's generate the first version of the essay.

In [5]:
mergesort_code = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama-3.3-70b-versatile"
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": mergesort_code
    }
)

In [6]:
display_markdown(mergesort_code, raw=True)

**Merge Sort Algorithm Implementation**
======================================

### Overview

Merge sort is a divide-and-conquer algorithm that splits a list of elements into two halves, recursively sorts each half, and then merges the sorted halves.

### Code
```python
def merge_sort(arr):
    """
    Sorts an array using the merge sort algorithm.

    Args:
    arr (list): The input array to be sorted.

    Returns:
    list: The sorted array.
    """
    # Base case: If the array has 1 or fewer elements, it is already sorted
    if len(arr) <= 1:
        return arr

    # Divide the array into two halves
    mid = len(arr) // 2
    left_half = arr[:mid]
    right_half = arr[mid:]

    # Recursively sort each half
    left_half = merge_sort(left_half)
    right_half = merge_sort(right_half)

    # Merge the sorted halves
    return merge(left_half, right_half)


def merge(left, right):
    """
    Merges two sorted arrays into a single sorted array.

    Args:
    left (list): The first sorted array.
    right (list): The second sorted array.

    Returns:
    list: The merged sorted array.
    """
    merged = []
    left_index = 0
    right_index = 0

    # Merge smaller elements first
    while left_index < len(left) and right_index < len(right):
        if left[left_index] <= right[right_index]:
            merged.append(left[left_index])
            left_index += 1
        else:
            merged.append(right[right_index])
            right_index += 1

    # Append any remaining elements
    merged.extend(left[left_index:])
    merged.extend(right[right_index:])

    return merged


# Example usage
if __name__ == "__main__":
    arr = [64, 34, 25, 12, 22, 11, 90]
    sorted_arr = merge_sort(arr)
    print("Original array:", arr)
    print("Sorted array:", sorted_arr)
```

### Explanation

1. The `merge_sort` function takes an input array `arr` and recursively divides it into two halves until the base case is reached (i.e., the array has 1 or fewer elements).
2. The `merge` function merges two sorted arrays `left` and `right` into a single sorted array `merged`.
3. The `merge_sort` function returns the sorted array by recursively sorting each half and then merging the sorted halves using the `merge` function.
4. In the example usage, an unsorted array `arr` is created and passed to the `merge_sort` function. The sorted array is then printed to the console.

### Time Complexity

* Best-case time complexity: O(n log n)
* Average-case time complexity: O(n log n)
* Worst-case time complexity: O(n log n)

### Space Complexity

* The space complexity is O(n) because the `merge_sort` function creates temporary arrays to store the sorted halves.

## Reflection Step

Now, let's allow the LLM to reflect on its outputs by defining another system prompt. This system prompt will tell the LLM to act as Andrej Karpathy, computer scientist and Deep Learning wizard.

>To be honest, I don't think the fact of acting like Andrej Karpathy will influence the LLM outputs, but it was fun :)

<img src="../img/karpathy.png" alt="Alt text" width="500"/>

In [7]:
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are Andrej Karpathy, an experienced computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

The user message, in this case,  is the essay generated in the previous step. We simply add the `mergesort_code` to the `reflection_chat_history`.

In [8]:
reflection_chat_history.append(
    {
        "role": "user",
        "content": mergesort_code
    }
)

Now, let's generate a critique to the Python code.

In [9]:
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model="llama-3.3-70b-versatile"
).choices[0].message.content

In [10]:
display_markdown(critique, raw=True)

## Code Review

### Overall Structure

The overall structure of the code is well-organized and easy to follow. The use of Markdown headers and sections makes the code review process smooth.

### Merge Sort Algorithm Implementation

The implementation of the merge sort algorithm is correct and follows the standard approach. However, there are a few minor improvements that can be made:

*   The `merge_sort` function does not handle the case where the input array is `None`. Adding a simple check at the beginning of the function can handle this case.
*   The `merge` function uses a while loop to merge the two sorted arrays. This approach is correct, but it can be improved by using a more Pythonic approach, such as using the built-in `sorted` function with a custom key function.

### Code Improvements

Here are some code improvements that can be made:

*   **Type Hints**: Add type hints to the function parameters and return types to make the code more readable and self-documenting.
*   **Docstrings**: Add docstrings to the functions to provide a brief description of what each function does.
*   **Error Handling**: Add error handling to handle edge cases, such as an empty input array or a `None` input.

### Updated Code

Here is the updated code with the suggested improvements:

```python
def merge_sort(arr: list) -> list:
    """
    Sorts an array using the merge sort algorithm.

    Args:
    arr (list): The input array to be sorted.

    Returns:
    list: The sorted array.

    Raises:
    TypeError: If the input array is None.
    """
    if arr is None:
        raise TypeError("Input array cannot be None")

    # Base case: If the array has 1 or fewer elements, it is already sorted
    if len(arr) <= 1:
        return arr

    # Divide the array into two halves
    mid = len(arr) // 2
    left_half = arr[:mid]
    right_half = arr[mid:]

    # Recursively sort each half
    left_half = merge_sort(left_half)
    right_half = merge_sort(right_half)

    # Merge the sorted halves
    return merge(left_half, right_half)


def merge(left: list, right: list) -> list:
    """
    Merges two sorted arrays into a single sorted array.

    Args:
    left (list): The first sorted array.
    right (list): The second sorted array.

    Returns:
    list: The merged sorted array.
    """
    merged = []
    left_index = 0
    right_index = 0

    # Merge smaller elements first
    while left_index < len(left) and right_index < len(right):
        if left[left_index] <= right[right_index]:
            merged.append(left[left_index])
            left_index += 1
        else:
            merged.append(right[right_index])
            right_index += 1

    # Append any remaining elements
    merged.extend(left[left_index:])
    merged.extend(right[right_index:])

    return merged


# Example usage
if __name__ == "__main__":
    arr = [64, 34, 25, 12, 22, 11, 90]
    sorted_arr = merge_sort(arr)
    print("Original array:", arr)
    print("Sorted array:", sorted_arr)
```

### Time and Space Complexity

The time complexity of the updated code remains the same:

*   Best-case time complexity: O(n log n)
*   Average-case time complexity: O(n log n)
*   Worst-case time complexity: O(n log n)

The space complexity also remains the same:

*   The space complexity is O(n) because the `merge_sort` function creates temporary arrays to store the sorted halves.

Overall, the updated code is more readable, maintainable, and efficient. It also handles edge cases and provides a clear description of what each function does.

Finally, we just need to add this *critique* to the `generation_chat_history`, in this case, as the `user` role.

In [11]:
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

## Generation Step (II)

In [12]:
essay = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama-3.3-70b-versatile"
).choices[0].message.content

In [13]:
display_markdown(essay, raw=True)

## Code Review Response

### Introduction

Thank you for taking the time to review the code. Your feedback is greatly appreciated, and I'll do my best to address each point and provide an updated version of the code.

### Code Structure and Improvements

I agree with your assessment of the overall structure of the code. The use of Markdown headers and sections makes the code easy to read and understand.

Regarding the implementation of the merge sort algorithm, I appreciate your suggestion to add a check for `None` input at the beginning of the `merge_sort` function. This will help prevent potential errors and make the code more robust.

I also agree with your suggestion to add type hints and docstrings to the functions. This will make the code more readable and self-documenting.

### Updated Code

Here is the updated code with the suggested improvements:
```python
def merge_sort(arr: list) -> list:
    """
    Sorts an array using the merge sort algorithm.

    Args:
    arr (list): The input array to be sorted.

    Returns:
    list: The sorted array.

    Raises:
    TypeError: If the input array is None.
    ValueError: If the input array is empty.
    """
    if arr is None:
        raise TypeError("Input array cannot be None")
    if len(arr) == 0:
        raise ValueError("Input array cannot be empty")

    # Base case: If the array has 1 or fewer elements, it is already sorted
    if len(arr) <= 1:
        return arr

    # Divide the array into two halves
    mid = len(arr) // 2
    left_half = arr[:mid]
    right_half = arr[mid:]

    # Recursively sort each half
    left_half = merge_sort(left_half)
    right_half = merge_sort(right_half)

    # Merge the sorted halves
    return merge(left_half, right_half)


def merge(left: list, right: list) -> list:
    """
    Merges two sorted arrays into a single sorted array.

    Args:
    left (list): The first sorted array.
    right (list): The second sorted array.

    Returns:
    list: The merged sorted array.
    """
    merged = []
    left_index = 0
    right_index = 0

    # Merge smaller elements first
    while left_index < len(left) and right_index < len(right):
        if left[left_index] <= right[right_index]:
            merged.append(left[left_index])
            left_index += 1
        else:
            merged.append(right[right_index])
            right_index += 1

    # Append any remaining elements
    merged.extend(left[left_index:])
    merged.extend(right[right_index:])

    return merged


# Example usage
if __name__ == "__main__":
    arr = [64, 34, 25, 12, 22, 11, 90]
    sorted_arr = merge_sort(arr)
    print("Original array:", arr)
    print("Sorted array:", sorted_arr)
```

### Time and Space Complexity

The time complexity of the updated code remains the same:

*   Best-case time complexity: O(n log n)
*   Average-case time complexity: O(n log n)
*   Worst-case time complexity: O(n log n)

The space complexity also remains the same:

*   The space complexity is O(n) because the `merge_sort` function creates temporary arrays to store the sorted halves.

### Additional Suggestions

To further improve the code, here are a few additional suggestions:

*   Consider adding a `main` function to encapsulate the example usage code. This will make the code more modular and easier to test.
*   Consider using a more Pythonic approach to merge the sorted arrays, such as using the `sorted` function with a custom key function.
*   Consider adding more test cases to ensure the code works correctly for different input scenarios.

I hope this updated code meets your requirements. Please let me know if you have any further suggestions or feedback.

## And the iteration starts again ...

After **Generation Step (II)** the corrected Python code will be received, once again, by Karpathy. Then, the LLM will reflect on the corrected output, suggesting further improvements and the loop will go, over and over for a number **n** of total iterations.

> There's another possibility. Suppose the Reflection step can't find any further improvement. In this case, we can tell the LLM to output some stop string, like "OK" or "Good" that means the process can be stopped. However, we are going to follow the first approach, that is, iterating for a fixed number of times.

## Implementing a class

Now that you understand the underlying loop of the Reflection Agent, let's implement this agent as a class.

In [14]:
class ReflectionAgent:
    def __init__(self, model="llama-3.3-70b-versatile"):
        self.model = model

    def run(self, user_msg, generation_system_prompt, reflection_system_prompt, n_steps=3, verbose=0):
        # 1. Generation phase
        gen_history = [
            {"role": "system", "content": generation_system_prompt},
            {"role": "user", "content": user_msg}
        ]

        # Generate initial response
        current_response = client.chat.completions.create(
            messages=gen_history,
            model=self.model
        ).choices[0].message.content

        for i in range(n_steps):
            if verbose:
                print(f"--- Iteration {i+1} ---")

            # 2. Reflection phase
            ref_history = [
                {"role": "system", "content": reflection_system_prompt},
                {"role": "user", "content": current_response}
            ]

            critique = client.chat.completions.create(
                messages=ref_history,
                model=self.model
            ).choices[0].message.content

            if verbose:
                print("Critique received.")

            # 3. Refinement phase
            gen_history.append({"role": "assistant", "content": current_response})
            gen_history.append({"role": "user", "content": critique})

            current_response = client.chat.completions.create(
                messages=gen_history,
                model=self.model
            ).choices[0].message.content

        return current_response

In [15]:
agent = ReflectionAgent()

In [16]:
generation_system_prompt = "You are a Python programmer tasked with generating high quality Python code"

reflection_system_prompt = "You are Andrej Karpathy, an experienced computer scientist"

user_msg = "Generate a Python implementation of the Merge Sort algorithm"

In [18]:
final_response = agent.run(
    user_msg=user_msg,
    generation_system_prompt=generation_system_prompt,
    reflection_system_prompt=reflection_system_prompt,
    n_steps=3, # Reduced n_steps to avoid hitting the token limit
    verbose=1,
)

--- Iteration 1 ---
Critique received.
--- Iteration 2 ---
Critique received.
--- Iteration 3 ---
Critique received.


## Final result

In [19]:
display_markdown(final_response, raw=True)

**Code Review and Final Improvements**
=====================================

The final version of the code provided is a robust and efficient implementation of the merge sort algorithm in Python. The addition of type hints, error handling, and comprehensive test cases makes the code more maintainable, efficient, and easier to understand.

### Final Suggestions for Improvement

The following suggestions can further improve the code:

1.  **Using a More Advanced Testing Framework**: Consider using a more advanced testing framework such as `unittest` to make the test cases more comprehensive and easier to maintain.
2.  **Encapsulating Example Usage and Test Cases**: The `main` function can be used to encapsulate the example usage and test cases, making the code more modular and reusable.
3.  **Support for Sorting Arrays of Custom Objects**: Adding support for sorting arrays of custom objects can further enhance the code. This can be achieved by using a custom comparison function or by implementing the `__lt__` method in the custom object class.

### Time and Space Complexity

*   **Time Complexity**: The time complexity of the merge sort algorithm is O(n log n), where n is the number of elements in the input array. This is because the algorithm recursively divides the array into two halves until each half has one or zero elements, and then merges the sorted halves.
*   **Space Complexity**: The space complexity of the merge sort algorithm is O(n), where n is the number of elements in the input array. This is because the algorithm creates temporary arrays to store the sorted halves during the merging process.

Here's the code with some final suggestions implemented:

```python
import unittest

def merge_sort(arr: list) -> list:
    """
    Sorts an array using the merge sort algorithm.

    Args:
        arr (list): The input array to be sorted.

    Returns:
        list: The sorted array.

    Raises:
        TypeError: If the input array is not a list or contains non-comparable elements.
    """
    if not isinstance(arr, list):
        raise TypeError("Input array must be a list")

    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left_half = arr[:mid]
    right_half = arr[mid:]

    left_half = merge_sort(left_half)
    right_half = merge_sort(right_half)

    return merge(left_half, right_half)


def merge(left: list, right: list) -> list:
    """
    Merges two sorted arrays into a single sorted array.

    Args:
        left (list): The first sorted array.
        right (list): The second sorted array.

    Returns:
        list: The merged sorted array.

    Raises:
        TypeError: If the input arrays contain non-comparable elements.
    """
    result = []
    left_index = 0
    right_index = 0

    while left_index < len(left) and right_index < len(right):
        try:
            if left[left_index] <= right[right_index]:
                result.append(left[left_index])
                left_index += 1
            else:
                result.append(right[right_index])
                right_index += 1
        except TypeError:
            raise TypeError("Input arrays contain non-comparable elements")

    result.extend(left[left_index:])
    result.extend(right[right_index:])

    return result


class TestMergeSort(unittest.TestCase):
    def test_sorting(self):
        # Test case: Sorting an array with duplicate elements
        arr = [4, 2, 9, 6, 2, 1, 8]
        sorted_arr = merge_sort(arr)
        self.assertEqual(sorted_arr, [1, 2, 2, 4, 6, 8, 9])

        # Test case: Sorting an array with negative numbers
        arr = [-5, 0, -2, 3, -1, 4]
        sorted_arr = merge_sort(arr)
        self.assertEqual(sorted_arr, [-5, -2, -1, 0, 3, 4])

        # Test case: Sorting an empty array
        arr = []
        sorted_arr = merge_sort(arr)
        self.assertEqual(sorted_arr, [])

        # Test case: Sorting an array with a single element
        arr = [5]
        sorted_arr = merge_sort(arr)
        self.assertEqual(sorted_arr, [5])

if __name__ == "__main__":
    unittest.main(argv=[''], verbosity=2, exit=False)

    # Example usage:
    arr = [64, 34, 25, 12, 22, 11, 90]
    sorted_arr = merge_sort(arr)
    print("Original array:", arr)
    print("Sorted array:", sorted_arr)

```

The improved version of the code maintains the same organization and structure as the original code but includes some additional test cases and error handling to make it more robust.

### Code Organization

The improved code is well-organized into separate functions for the merge sort algorithm and the merging of sorted arrays. The `TestMergeSort` class encapsulates the test cases, making the code more modular and reusable.

### Time and Space Complexity

The time complexity of the merge sort algorithm is O(n log n), where n is the number of elements in the input array. The space complexity is O(n), where n is the number of elements in the input array.

### Future Improvements

Some potential future improvements to the code include:

*   **Implementing a more efficient sorting algorithm**: Depending on the specific requirements of the application, a more efficient sorting algorithm such as quicksort or heapsort may be more suitable.
*   **Adding support for sorting arrays of custom objects**: This can be achieved by using a custom comparison function or by implementing the `__lt__` method in the custom object class.
*   **Improving the error handling**: The code can be improved by adding more informative error messages and handling edge cases such as sorting an array with a single element or an empty array.

Overall, the improved code is a robust and efficient implementation of the merge sort algorithm in Python. It includes comprehensive test cases and uses a more advanced testing framework to ensure the correctness of the algorithm.